In [ ]:
import pandas as pd
from model_tuner import loadObjects
from model_metrics import summarize_model_performance

## Paths

In [ ]:
data_path = "../data/processed"

In [ ]:
from eda_toolkit import ensure_directory
import os  # import operating system for dir

base_path = os.path.join(os.pardir)

# Go up one level from 'notebooks' to parent directory,
# then into the 'data' folder
data_path = os.path.join(os.pardir, "data/processed")

# create image paths
image_path_png = os.path.join(base_path, "images", "png_images")
image_path_svg = os.path.join(base_path, "images", "svg_images")

# Use the function to ensure'data' directory exists
ensure_directory(data_path)
ensure_directory(image_path_png)
ensure_directory(image_path_svg)

In [ ]:
import json
from pathlib import Path

pred_dir = Path("../models/predictions/full_text_clean")

X = pd.read_parquet("../data/processed/X.parquet")
y = pd.read_parquet("../data/processed/y.parquet").squeeze()

In [ ]:
from core.model_registry import available, load_all, load_model

available()  # should now show cat_feats_and_text, cat_text_only, cat, lr across all three experiments

In [ ]:
from core.model_registry import best_per_algo, load_best_per_algo

best_per_algo(metric="valid Average Precision")
champs = load_best_per_algo(metric="valid Average Precision")

In [ ]:
champs

In [ ]:
model_catboost = champs["cat_outcome"]
model_catboost_no_sex = champs["cat_outcome_no_sex"]
model_xgboost = champs["xgb_outcome"]
model_rf = champs["rf_outcome"]
model_lr = champs["lr_outcome"]

In [ ]:
model_titles = ["CatBoost", "CatBoost_No_Sex", "XGBoost", "Random Forest", "Logistic Regression"]
models = [model_catboost, model_catboost_no_sex, model_xgboost, model_rf, model_lr,]

In [ ]:
thresholds = {
    "Logistic Regression": next(iter(model_lr.threshold.values())),
    "Random Forest Classifier": next(iter(model_rf.threshold.values())),
    "XGBoost": next(iter(model_xgboost.threshold.values())),
    "CatBoost": next(iter(model_catboost.threshold.values())),
    "CatBoost_No_Sex": next(iter(model_catboost_no_sex.threshold.values())),
}

In [ ]:
X_valid, y_valid = model_lr.get_valid_data(X, y)
X_test, y_test = model_lr.get_test_data(X, y)
y_test = y_test["outcome"]

In [ ]:
from model_metrics import summarize_model_performance

model_performance = summarize_model_performance(
    model=models,
    model_title=model_titles,
    X=X_test,
    y=y_test,
    model_type="classification",
    return_df=True,
    model_threshold=thresholds,
)

model_performance

In [ ]:
from model_metrics import show_roc_curve

show_roc_curve(
    model=models,
    X=X_test,
    y=y_test,
    model_title=model_titles,
    decimal_places=2,
    curve_kwgs={
        "CatBoost": {"color": "green", "linewidth": 1},
        "CatBoost_No_Sex": {"color": "orange", "linewidth": 1},
        "XGBoost": {"color": "purple", "linewidth": 1},
        "Random Forest": {"color": "black", "linewidth": 1},
        "Logistic Regression": {"color": "blue", "linewidth": 1},
    },
    linestyle_kwgs={"color": "red", "linestyle": "--"},
    title="ROC Curves: Logistic Regression and Random Forest",
    overlay=True,
)